<a id="top"></a>

# Lab L0206: structured output and defensive parsing

```yaml
title: "Lab L0206: structured output and defensive parsing"
keywords: structured output, json, defensive parsing, regex fallback, enum validation, fail loudly, lab
estimated duration: 35
```

> **Lesson:** L02. See [objectives.md](../../../../docs/origin/lesson_roadmaps/L02/objectives.md). Solutions: `L0206_lab_solutions.ipynb`. Write a defensive parser, prove it on crafted hard cases, then point it at a **live** model reply — set `ANTHROPIC_API_KEY` (copy `.env.example` to `.env`) before running. Clear outputs before committing.
>
> **You'll walk out able to:** write a defensive JSON parser; validate keys and enums; explain why structured output matters before agents exist.

## Contents

- [1. Setup](#1-setup)
- [2. Problem 1 — The defensive parser](#2-problem-1--the-defensive-parser)
- [3. Problem 2 — Validate the contract](#3-problem-2--validate-the-contract)
- [4. Problem 3 — Run it over crafted cases, then a live reply](#4-problem-3--run-it-over-crafted-cases-then-a-live-reply)
- [5. Problem 4 — Why structured output, before agents?](#5-problem-4--why-structured-output-before-agents)
- [6. Self-check](#6-self-check)

## 1. Setup

In [ ]:
import json
import re
from typing import Any

from fluffy_potato_curriculum.potato_llm import AnthropicClient, Message, PotatoLLMClient

ALLOWED_STATUS = {"todo", "in_progress", "done"}

# Crafted hard cases for the same 'extract a task as JSON' prompt. A well-behaved live
# model rarely emits the prose-wrapped or malformed shapes, so we keep them by hand to
# prove the parser's fallback and fail-loudly paths — that is the whole point of
# defensive parsing.
REPLIES: dict[str, str] = {
    "clean": '{"title": "Ship the L02 lab", "status": "in_progress", "assignee": "Sam"}',
    "prose_wrapped": (
        "Sure! Here is the task as JSON:\n\n"
        '{"title": "Fix the parser", "status": "todo", "assignee": null}'
    ),
    "malformed": "status: done, title: forgot the braces entirely",
}

# Prompt + live client for the real-reply check in Problem 3.
EXTRACT_PROMPT = (
    "Extract the task described below as a single JSON object with keys "
    "title (string), status (one of: todo, in_progress, done), and assignee "
    "(string or null). Return only the JSON.\n\nTask: "
)
client: PotatoLLMClient = AnthropicClient()
print(f"setup ready (live client: {client.model})")

[↑ Back to top](#top)

## 2. Problem 1 — The defensive parser

Implement `parse_json_object` with the three-step discipline:

1. `json.loads` the whole reply;
2. on failure, regex out the first `{...}` block (use `re.DOTALL`) and retry;
3. on failure again, **raise `ValueError`** including the raw reply.

Never return a silent empty dict — fail loudly.

In [ ]:
def parse_json_object(reply: str) -> dict[str, Any]:
    try:
        return json.loads(reply)
    except json.JSONDecodeError:
        pass

    match = re.search(r"\{.*\}", reply, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            pass

    raise ValueError(f"Could not parse a JSON object from reply:\n{reply!r}")


print(parse_json_object(REPLIES["clean"]))
print(parse_json_object(REPLIES["prose_wrapped"]))  # regex fallback should save this

[↑ Back to top](#top)

## 3. Problem 2 — Validate the contract

A successful parse isn't a honored contract. Implement `validate_task`: return a list of problems (empty == valid). Check that `title` and `status` keys are present, and that `status` is one of `ALLOWED_STATUS`. A `null`/missing `assignee` is allowed.

In [ ]:
def validate_task(record: dict[str, Any]) -> list[str]:
    problems: list[str] = []
    for key in ("title", "status"):
        if key not in record:
            problems.append(f"missing key: {key}")
    status = record.get("status")
    if status is not None and status not in ALLOWED_STATUS:
        problems.append(f"status {status!r} not in {sorted(ALLOWED_STATUS)}")
    return problems


print(validate_task({"title": "x", "status": "done"}))  # expect []
print(validate_task({"title": "x", "status": "blocked"}))  # expect a status problem
print(validate_task({"status": "todo"}))  # expect a missing-title problem

[↑ Back to top](#top)

## 4. Problem 3 — Run it over crafted cases, then a live reply

First, loop over the three crafted `REPLIES`. Parse each (catch `ValueError`, print a loud notice), then validate the ones that parsed. Confirm: `clean` is fine, `prose_wrapped` is salvaged, `malformed` fails loudly.

Then point the **same** parser at a live model reply — ask the model to extract a task, and run `parse_json_object` + `validate_task` on whatever comes back. A real reply is usually clean, which is exactly why the crafted cases above still earn their keep: defensive parsing is for the bad reply you'll eventually get, not the good one you usually get.

In [ ]:
for name, reply in REPLIES.items():
    print(f"=== {name} ===")
    try:
        record = parse_json_object(reply)
    except ValueError as err:
        print("PARSE FAILED LOUDLY:", err)
        print()
        continue
    print("parsed:", record)
    print("problems:", validate_task(record) or "none")
    print()

# The same discipline, now on a live reply.
description = "Ship the L02 lab — Sam is on it and it's underway."
live_reply = (
    await client.achat(
        [Message.user(EXTRACT_PROMPT + description)], max_tokens=120, temperature=0.0
    )
).text
print("=== live ===")
print("raw:", live_reply.replace(chr(10), " ⏎ "))
try:
    record = parse_json_object(live_reply)
    print("parsed:", record)
    print("problems:", validate_task(record) or "none")
except ValueError as err:
    print("PARSE FAILED LOUDLY:", err)

[↑ Back to top](#top)

## 5. Problem 4 — Why structured output, before agents?

In 2–3 sentences: why does structured output matter *before* you ever build an agent? Name one later thing it unlocks (evals, tool calling, pipelines), and state what L07 adds on top of the prompt-only approach you used here.

> **Answer.** Structured output makes the model's response *programmatic* — a dict your code can index — instead of *conversational* prose you have to read, and that's the precondition for everything downstream: evals (L13), tool calling (L07), and multi-step pipelines all need machine-readable output. That's why it matters before any agent exists. L07 adds tool-use-as-schema to *force* schema-conformant output, but the defensive-parsing discipline you used here still applies — tool-call arguments can be malformed too.

## 6. Self-check

Diff against `L0206_lab_solutions.ipynb`. Done when `prose_wrapped` is salvaged, `malformed` raises loudly, `validate_task` flags the out-of-set status, and your parser cleanly handles the live reply too.

[↑ Back to top](#top)